In [25]:
import tensorflow as tf
import numpy as np
import pygad
from typing import Optional, Dict, Tuple, List
import warnings
warnings.filterwarnings('ignore')

def hydrolysis_prediction_optimization(
    cellulose: float,
    hemicellulose: float, 
    lignin: float,
    solid_loading: Optional[float] = None,
    enzyme_loading: Optional[float] = None,
    reaction_time: Optional[float] = None,
    model_path: str = r"c:\Users\audec\OneDrive\Ethanol-AI\BEPE FAPESP\Genetic ANNs\Straw\Hydrolysis\champion_ann_strategy1_32_32_16.h5"
) -> Dict:
    """
    Função inteligente para predição de hidrólise enzimática com otimização reversa.
    
    Parâmetros obrigatórios:
    - cellulose: Percentual de celulose na biomassa (0-1)
    - hemicellulose: Percentual de hemicelulose na biomassa (0-1)  
    - lignin: Percentual de lignina na biomassa (0-1)
    
    Parâmetros opcionais (processo):
    - solid_loading: Carregamento de sólidos em g/L (50-250)
    - enzyme_loading: Carregamento de enzima em g/L (0.01-1.5)
    - reaction_time: Tempo de reação em horas (1-96)
    - model_path: Caminho para o modelo ANN treinado
    
    Retorna:
    - Dict com resultados de predição e/ou otimização
    """
    
    # Carregar o modelo ANN
    try:
        # Tentar carregar com objetos customizados
        model = tf.keras.models.load_model(model_path, custom_objects={'mse': 'mean_squared_error'})
        print(f"✅ Modelo carregado: {model_path}")
    except Exception as e1:
        try:
            # Fallback: carregar sem objetos customizados
            model = tf.keras.models.load_model(model_path, compile=False)
            print(f"✅ Modelo carregado (sem compilação): {model_path}")
        except Exception as e2:
            return {"error": f"Erro ao carregar modelo: {str(e1)} | Fallback: {str(e2)}"}
    
    # Preparar inputs conhecidos (composição da biomassa)
    known_inputs = np.array([[cellulose, hemicellulose, lignin]])
    
    # Verificar quais inputs de processo foram fornecidos
    provided_inputs = []
    missing_inputs = []
    input_names = ['solid_loading', 'enzyme_loading', 'reaction_time']
    input_values = [solid_loading, enzyme_loading, reaction_time]
    
    for name, value in zip(input_names, input_values):
        if value is not None:
            provided_inputs.append((name, value))
        else:
            missing_inputs.append(name)
    
    print(f"📊 Inputs fornecidos: {[name for name, _ in provided_inputs]}")
    print(f"🔍 Inputs a otimizar: {missing_inputs}")
    
    # CASO 1: Todos os 6 inputs fornecidos - Predição direta
    if len(missing_inputs) == 0:
        print("\n🎯 Modo: Predição Direta")
        full_inputs = np.concatenate([
            known_inputs, 
            np.array([[solid_loading, enzyme_loading, reaction_time]])
        ], axis=1)
        
        try:
            predictions = model.predict(full_inputs, verbose=0)
        except Exception as e:
            return {"error": f"Erro na predição do modelo: {str(e)}"}
        
        return {
            "mode": "direct_prediction",
            "inputs": {
                "cellulose": cellulose,
                "hemicellulose": hemicellulose, 
                "lignin": lignin,
                "solid_loading": solid_loading,
                "enzyme_loading": enzyme_loading,
                "reaction_time": reaction_time
            },
            "predictions": {
                "glucose": float(predictions[0][0]),
                "xylose": float(predictions[0][1]),
                "cellobiose": float(predictions[0][2])
            }
        }
    
    # CASO 2: Alguns inputs faltando - Otimização reversa
    else:
        print(f"\n🔧 Modo: Otimização Reversa ({len(missing_inputs)} inputs a otimizar)")
        
        # Definir ranges para otimização
        ranges = {
            'solid_loading': {'low': 50, 'high': 250},
            'enzyme_loading': {'low': 0.01, 'high': 1.5},
            'reaction_time': {'low': 1, 'high': 96}
        }
        
        # Criar gene_space apenas para inputs faltando
        gene_space = [ranges[name] for name in missing_inputs]
        
        # Função de fitness para maximizar glicose
        def fitness_func(ga_instance, solution, solution_idx):
            # Reconstruir array completo dos inputs de processo
            process_inputs = [0, 0, 0]  # [solid_loading, enzyme_loading, reaction_time]
            
            # Preencher valores fornecidos
            sol_idx = 0  # Renomeado para evitar conflito com parâmetro
            for i, name in enumerate(input_names):
                if name in [p[0] for p in provided_inputs]:
                    # Usar valor fornecido
                    value = next(p[1] for p in provided_inputs if p[0] == name)
                    process_inputs[i] = value
                else:
                    # Usar valor da otimização
                    process_inputs[i] = solution[sol_idx]
                    sol_idx += 1
            
            # Criar input completo
            full_inputs = np.concatenate([
                known_inputs,
                np.array([process_inputs])
            ], axis=1).astype(np.float32)
            
            # Predição
            try:
                predictions = model.predict(full_inputs, verbose=0)
                glucose_prediction = predictions[0][0]
            except Exception as e:
                # Se houver erro na predição, retornar fitness muito baixo
                return -1000.0
            
            # Fitness = maximizar glicose (com pequena penalidade para estabilidade)
            fitness = glucose_prediction - 0.001 * np.sum(np.square(solution))
            
            return float(fitness)
        
        # Configurar algoritmo genético
        try:
            ga_instance = pygad.GA(
                num_generations=30,
                num_parents_mating=20,
                fitness_func=fitness_func,
                sol_per_pop=40,
                num_genes=len(missing_inputs),
                gene_space=gene_space,
                mutation_type="random",
                mutation_percent_genes=15,
                keep_parents=3,
                crossover_type="single_point",  # Mudando para single_point por compatibilidade
                parent_selection_type="sss",
                random_seed=42,
                gene_type=float,  # Forçar tipo float
                suppress_warnings=True
            )
        except Exception as e:
            # Fallback para versões mais antigas do PyGAD
            ga_instance = pygad.GA(
                num_generations=30,
                num_parents_mating=20,
                fitness_func=fitness_func,
                sol_per_pop=40,
                num_genes=len(missing_inputs),
                gene_space=gene_space,
                mutation_type="random",
                mutation_percent_genes=15,
                keep_parents=3,
                crossover_type="single_point",
                parent_selection_type="sss",
                random_seed=42,
                gene_type=float
            )
        
        # Executar otimização
        print("🚀 Executando otimização genética...")
        ga_instance.run()
        
        # Obter melhor solução
        solution, solution_fitness, _ = ga_instance.best_solution()
        
        # Reconstruir inputs completos
        optimized_inputs = {}
        sol_idx = 0  # Renomeado para evitar conflito
        for i, name in enumerate(input_names):
            if name in [p[0] for p in provided_inputs]:
                optimized_inputs[name] = next(p[1] for p in provided_inputs if p[0] == name)
            else:
                optimized_inputs[name] = solution[sol_idx]
                sol_idx += 1
        
        # Fazer predição final com valores otimizados
        final_process_inputs = [
            optimized_inputs['solid_loading'],
            optimized_inputs['enzyme_loading'], 
            optimized_inputs['reaction_time']
        ]
        
        final_full_inputs = np.concatenate([
            known_inputs,
            np.array([final_process_inputs])
        ], axis=1)
        
        try:
            final_predictions = model.predict(final_full_inputs, verbose=0)
        except Exception as e:
            return {"error": f"Erro na predição final: {str(e)}"}
        
        return {
            "mode": "reverse_optimization",
            "optimized_for": missing_inputs,
            "provided_inputs": {name: value for name, value in provided_inputs},
            "optimized_inputs": {
                name: optimized_inputs[name] for name in missing_inputs
            },
            "all_inputs": {
                "cellulose": cellulose,
                "hemicellulose": hemicellulose,
                "lignin": lignin,
                "solid_loading": optimized_inputs['solid_loading'],
                "enzyme_loading": optimized_inputs['enzyme_loading'],
                "reaction_time": optimized_inputs['reaction_time']
            },
            "predictions": {
                "glucose": float(final_predictions[0][0]),
                "xylose": float(final_predictions[0][1]),
                "cellobiose": float(final_predictions[0][2])
            },
            "fitness_score": float(solution_fitness)
        }

# Função auxiliar para exibir resultados de forma organizada
def display_results(results: Dict):
    """Exibe os resultados de forma organizada"""
    print("\n" + "="*60)
    print("📊 RESULTADOS DA ANÁLISE DE HIDRÓLISE")
    print("="*60)
    
    if "error" in results:
        print(f"❌ Erro: {results['error']}")
        return
    
    mode = results["mode"]
    
    if mode == "direct_prediction":
        print("🎯 Modo: Predição Direta")
        print("\n📥 Inputs utilizados:")
        for key, value in results["inputs"].items():
            print(f"  • {key}: {value:.4f}")
            
    else:  # reverse_optimization
        print("🔧 Modo: Otimização Reversa")
        print(f"\n🔍 Parâmetros otimizados: {results['optimized_for']}")
        
        if results["provided_inputs"]:
            print("\n📥 Inputs fornecidos:")
            for key, value in results["provided_inputs"].items():
                print(f"  • {key}: {value:.4f}")
        
        print("\n🎯 Inputs otimizados:")
        for key, value in results["optimized_inputs"].items():
            print(f"  • {key}: {value:.4f}")
        
        print(f"\n💯 Score de fitness: {results['fitness_score']:.4f}")
    
    print("\n📊 Predições finais:")
    for key, value in results["predictions"].items():
        print(f"  • {key}: {value:.4f}")
    
    print("="*60)

In [28]:
# TESTE 1: Predição Direta
print("TESTE 1: Predição Direta")
results1 = hydrolysis_prediction_optimization(
    cellulose=0.6,
    hemicellulose=0.1, 
    lignin=0.3,
    solid_loading=150.0,
    enzyme_loading=0.5,
    reaction_time=48.0
)
display_results(results1)

print("\n" + "="*80 + "\n")

# TESTE 2: Otimização (faltando reaction_time)
print("TESTE 2: Otimização - Faltando reaction_time")
results2 = hydrolysis_prediction_optimization(
    cellulose=0.4,
    hemicellulose=0.25,
    lignin=0.35,
    solid_loading=120.0,
    enzyme_loading=0.3
    # reaction_time será otimizado
)
display_results(results2)

TESTE 1: Predição Direta
✅ Modelo carregado: c:\Users\audec\OneDrive\Ethanol-AI\BEPE FAPESP\Genetic ANNs\Straw\Hydrolysis\champion_ann_strategy1_32_32_16.h5
📊 Inputs fornecidos: ['solid_loading', 'enzyme_loading', 'reaction_time']
🔍 Inputs a otimizar: []

🎯 Modo: Predição Direta

📊 RESULTADOS DA ANÁLISE DE HIDRÓLISE
🎯 Modo: Predição Direta

📥 Inputs utilizados:
  • cellulose: 0.6000
  • hemicellulose: 0.1000
  • lignin: 0.3000
  • solid_loading: 150.0000
  • enzyme_loading: 0.5000
  • reaction_time: 48.0000

📊 Predições finais:
  • glucose: 51.3224
  • xylose: 57.9697
  • cellobiose: 74.6026


TESTE 2: Otimização - Faltando reaction_time
✅ Modelo carregado: c:\Users\audec\OneDrive\Ethanol-AI\BEPE FAPESP\Genetic ANNs\Straw\Hydrolysis\champion_ann_strategy1_32_32_16.h5
📊 Inputs fornecidos: ['solid_loading', 'enzyme_loading']
🔍 Inputs a otimizar: ['reaction_time']

🔧 Modo: Otimização Reversa (1 inputs a otimizar)
🚀 Executando otimização genética...

📊 RESULTADOS DA ANÁLISE DE HIDRÓLISE
🔧 

In [27]:
# Função específica para integração com Streamlit
def streamlit_hydrolysis_analysis(
    cellulose: float,
    hemicellulose: float, 
    lignin: float,
    solid_loading: float = 0.0,  # 0.0 indica não fornecido
    enzyme_loading: float = 0.0,  # 0.0 indica não fornecido  
    reaction_time: float = 0.0,   # 0.0 indica não fornecido
    model_path: str = r"c:\Users\audec\OneDrive\Ethanol-AI\BEPE FAPESP\Genetic ANNs\Straw\Hydrolysis\champion_ann_strategy1_32_32_16.h5"
) -> Dict:
    """
    Função adaptada para uso com Streamlit.
    
    No Streamlit, inputs não fornecidos terão valor 0.0 
    (que está fora dos ranges válidos).
    
    Parâmetros:
    - cellulose, hemicellulose, lignin: Composição da biomassa (0-1)
    - solid_loading: 0.0 = não fornecido, >0 = valor fornecido (50-250 g/L)
    - enzyme_loading: 0.0 = não fornecido, >0 = valor fornecido (0.01-1.5 g/L)  
    - reaction_time: 0.0 = não fornecido, >0 = valor fornecido (1-96 h)
    
    Returns:
    - Dict com resultados formatados para Streamlit
    """
    
    # Converter 0.0 para None para usar na função principal
    solid_loading_input = solid_loading if solid_loading > 0 else None
    enzyme_loading_input = enzyme_loading if enzyme_loading > 0 else None  
    reaction_time_input = reaction_time if reaction_time > 0 else None
    
    # Chamar função principal
    results = hydrolysis_prediction_optimization(
        cellulose=cellulose,
        hemicellulose=hemicellulose,
        lignin=lignin,
        solid_loading=solid_loading_input,
        enzyme_loading=enzyme_loading_input,
        reaction_time=reaction_time_input,
        model_path=model_path
    )
    
    # Adicionar informações específicas para Streamlit
    if "error" not in results:
        # Calcular quais inputs foram fornecidos vs otimizados
        provided_count = sum([
            1 for x in [solid_loading_input, enzyme_loading_input, reaction_time_input] 
            if x is not None
        ])
        
        results["streamlit_info"] = {
            "inputs_provided": provided_count,
            "inputs_optimized": 3 - provided_count,
            "analysis_type": "Predição Completa" if provided_count == 3 else f"Otimização de {3-provided_count} Parâmetro(s)"
        }
    
    return results

# Exemplo de uso para Streamlit
print("EXEMPLO DE USO PARA STREAMLIT:")
print("-" * 50)

# Simular input do Streamlit onde usuário só preencheu solid_loading
streamlit_results = streamlit_hydrolysis_analysis(
    cellulose=0.4,
    hemicellulose=0.25, 
    lignin=0.35,
    solid_loading=100.0,  # Usuário preencheu
    enzyme_loading=0.0,   # Não preenchido (será otimizado)
    reaction_time=0.0     # Não preenchido (será otimizado)
)

if "streamlit_info" in streamlit_results:
    info = streamlit_results["streamlit_info"]
    print(f"📊 Tipo de análise: {info['analysis_type']}")
    print(f"📥 Inputs fornecidos: {info['inputs_provided']}")
    print(f"🔧 Inputs otimizados: {info['inputs_optimized']}")

display_results(streamlit_results)

EXEMPLO DE USO PARA STREAMLIT:
--------------------------------------------------
✅ Modelo carregado: c:\Users\audec\OneDrive\Ethanol-AI\BEPE FAPESP\Genetic ANNs\Straw\Hydrolysis\champion_ann_strategy1_32_32_16.h5
📊 Inputs fornecidos: ['solid_loading']
🔍 Inputs a otimizar: ['enzyme_loading', 'reaction_time']

🔧 Modo: Otimização Reversa (2 inputs a otimizar)
🚀 Executando otimização genética...
📊 Tipo de análise: Otimização de 2 Parâmetro(s)
📥 Inputs fornecidos: 1
🔧 Inputs otimizados: 2

📊 RESULTADOS DA ANÁLISE DE HIDRÓLISE
🔧 Modo: Otimização Reversa

🔍 Parâmetros otimizados: ['enzyme_loading', 'reaction_time']

📥 Inputs fornecidos:
  • solid_loading: 100.0000

🎯 Inputs otimizados:
  • enzyme_loading: 1.4991
  • reaction_time: 95.9732

💯 Score de fitness: 47.8772

📊 Predições finais:
  • glucose: 57.0903
  • xylose: 46.2142
  • cellobiose: 74.3915
📊 Tipo de análise: Otimização de 2 Parâmetro(s)
📥 Inputs fornecidos: 1
🔧 Inputs otimizados: 2

📊 RESULTADOS DA ANÁLISE DE HIDRÓLISE
🔧 Modo: Ot

# 🚀 Instruções para Uso no Streamlit

## Para usar essa função no seu web app Streamlit:

### 1. **Criar arquivo separado** (ex: `hydrolysis_optimizer.py`)
Copie a função `streamlit_hydrolysis_analysis` e suas dependências para um arquivo Python separado.

### 2. **No seu app Streamlit**, use assim:

```python
import streamlit as st
from hydrolysis_optimizer import streamlit_hydrolysis_analysis

# Inputs do usuário
cellulose = st.number_input("Celulose (%)", min_value=0.0, max_value=1.0, format="%.3f")
hemicellulose = st.number_input("Hemicelulose (%)", min_value=0.0, max_value=1.0, format="%.3f")  
lignin = st.number_input("Lignina (%)", min_value=0.0, max_value=1.0, format="%.3f")

solid_loading = st.number_input("Initial Solids Loading (g/L)", min_value=0.0, max_value=300, format="%.2f")
enzyme_loading = st.number_input("Initial Enzyme Loading (g/L)", min_value=0.0, max_value=2.0, format="%.2f")
reaction_time = st.number_input("Reaction Time (h)", min_value=0.0, max_value=96.0, format="%.2f")

# Processamento
if st.button("Analisar Hidrólise"):
    results = streamlit_hydrolysis_analysis(
        cellulose, hemicellulose, lignin,
        solid_loading, enzyme_loading, reaction_time
    )
    
    # Exibir resultados
    if "error" not in results:
        st.success(f"✅ {results['streamlit_info']['analysis_type']}")
        
        # Mostrar predições
        st.subheader("📊 Predições")
        col1, col2, col3 = st.columns(3)
        
        with col1:
            st.metric("Glucose", f"{results['predictions']['glucose']:.2f}")
        with col2:
            st.metric("Xylose", f"{results['predictions']['xylose']:.2f}")  
        with col3:
            st.metric("Cellobiose", f"{results['predictions']['cellobiose']:.2f}")
        
        # Se houve otimização, mostrar parâmetros otimizados
        if results["mode"] == "reverse_optimization":
            st.subheader("🎯 Parâmetros Otimizados")
            for param, value in results["optimized_inputs"].items():
                st.write(f"**{param}**: {value:.4f}")
    else:
        st.error(results["error"])
```

### 3. **Arquivos necessários no diretório do Streamlit:**
- `champion_ann_strategy1_32_32_16.h5` (modelo treinado)
- `hydrolysis_optimizer.py` (funções exportadas)
- `streamlit_app.py` (seu app principal)

### 4. **Dependências** (requirements.txt):
```
streamlit
tensorflow
numpy
pygad
```